# PowerPlus — Historical Electricity + Household Metadata + Islamabad.csv


This notebook uses the two historical electricity files supplied for:

- House#41 — Islamabad
- House#42 — Islamabad

The electricity data is minute-level for approximately one year.

Pipeline:

**Minute electricity → daily consumption → household metadata → islamabad.csv historical weather → feature engineering → Decision Tree Regressor**

The weather is collected through the islamabad.csv


## 1. Install/import libraries

In [57]:
# If needed, run once:
# !pip install pandas numpy openpyxl requests scikit-learn matplotlib seaborn

import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load the two electricity files

In [58]:
house41_path = "islamabad_House41.csv"
house42_path = "islamabad_House42.csv"

h41 = pd.read_csv(house41_path)
h42 = pd.read_csv(house42_path)

h41["House"] = "House#41"
h42["House"] = "House#42"

print("House 41:", h41.shape)
print("House 42:", h42.shape)

display(h41.head())
display(h42.head())


House 41: (525600, 3)
House 42: (525600, 7)


,datetime,Usage (kW),House
0,2023-11-01,0.63,House#41
1,2023-11-01 00:01:00,0.64,House#41
2,2023-11-01 00:02:00,0.63,House#41
3,2023-11-01 00:03:00,0.63,House#41
4,2023-11-01 00:04:00,0.52,House#41


,datetime,Usage (kW),FF (kW),DR_AC (kW),BR_AC (kW),Kitchen (kW),House
0,2023-11-01,0.596267,0.181883,0.003917,0.006433,0.334333,House#42
1,2023-11-01 00:01:00,0.569950,0.181533,0.003700,0.005800,0.309100,House#42
2,2023-11-01 00:02:00,0.444650,0.181567,0.002733,0.003883,0.185500,House#42
3,2023-11-01 00:03:00,0.444883,0.181983,0.003567,0.004450,0.185433,House#42
4,2023-11-01 00:04:00,0.444917,0.181217,0.002667,0.004183,0.185717,House#42


## 3. Standardize the electricity data

In [59]:
def prepare_electricity(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df["Usage (kW)"] = pd.to_numeric(df["Usage (kW)"], errors="coerce")
    df = df.dropna(subset=["datetime", "Usage (kW)", "House"])
    df = df.sort_values("datetime")
    return df

h41 = prepare_electricity(h41)
h42 = prepare_electricity(h42)

print(h41["datetime"].min(), "to", h41["datetime"].max())
print(h42["datetime"].min(), "to", h42["datetime"].max())


2023-11-01 00:00:00 to 2024-10-30 00:00:00
2023-11-01 00:00:00 to 2024-10-30 00:00:00


## 4. Convert minute-level kW into daily electricity consumption

Your target is currently **power (kW)** measured every minute.

For daily demand/energy forecasting, we convert it to **daily energy (kWh)**.

Because readings are approximately one minute apart:

**daily kWh = sum of minute kW × 1/60**

This is more appropriate than simply averaging the kW values.


In [60]:
def minute_to_daily(df):
    x = df.copy()
    x["Date"] = x["datetime"].dt.floor("D")

    # Each observation represents approximately one minute.
    x["energy_kWh"] = x["Usage (kW)"] / 60.0

    daily = (
        x.groupby(["House", "Date"], as_index=False)
         .agg(
             Electricity_Consumption_kWh=("energy_kWh", "sum"),
             Average_Load_kW=("Usage (kW)", "mean"),
             Peak_Load_kW=("Usage (kW)", "max")
         )
    )
    return daily

daily41 = minute_to_daily(h41)
daily42 = minute_to_daily(h42)

consumption = pd.concat(
    [daily41, daily42],
    ignore_index=True
).sort_values(["House", "Date"])

print("Daily dataset:", consumption.shape)
display(consumption.head())


Daily dataset: (716, 5)


,House,Date,Electricity_Consumption_kWh,Average_Load_kW,Peak_Load_kW
0,House#41,2023-11-01,0.010500,0.63,0.63
1,House#41,2023-11-02,0.008500,0.51,0.51
2,House#41,2023-11-03,0.010500,0.63,0.63
3,House#41,2023-11-04,0.010167,0.61,0.61
4,House#41,2023-11-05,0.008333,0.50,0.50


## 5. Load household metadata

In [61]:
metadata = pd.read_excel("metadata_ultimate.xlsx")

metadata.columns = (
    metadata.columns
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Keep only the houses for which we currently have electricity data
metadata_subset = metadata[
    metadata["House"].isin(["House#41", "House#42"])
].copy()

display(metadata_subset)


,House,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),No. of Adults (14-60),No. of Seniors (above 60),No. of temporary residents,Property Area (Marla),...,Water Dispensers,Water Pumps,Electric Cooker,Electric heaters,Electric Irons,Sewing Machine,Microwave Ovens,Geysers,UPS,Other Electronic Devices
40,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0
41,House#42,Islamabad,Rented,6.0,6.0,0.0,5.0,1.0,0.0,8.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0


In [62]:
print("Consumption columns:")
print(consumption.columns.tolist())

print("\nMetadata columns:")
print(metadata_subset.columns.tolist())

Consumption columns:
['House', 'Date', 'Electricity_Consumption_kWh', 'Average_Load_kW', 'Peak_Load_kW']

Metadata columns:
['House', 'City', 'Owner/Rented', 'No. of people (Temp+Perm)', 'No. of Permanent residents', 'No. of Children (0-13)', 'No. of Adults (14-60)', 'No. of Seniors (above 60)', 'No. of temporary residents', 'Property Area (Marla)', 'Covered Area', 'No of Floors', 'Floor of Residency', 'Build year of house', 'Wapda Connection type', 'Average Ceiling Height ft', 'Ceiling Type', 'Roof Type', 'Flooring Type', 'Interior Wall', 'Exterior Wall', 'No. of rooms', 'Room Dimensions', 'Kitchen', 'Number of Washrooms', 'Number of Stores', 'Doors Type', 'Air Conditioners', 'Air Coolers', 'Refrigerators', 'Washing Machines', 'LED Bulbs', 'Tube Lights', 'Celling Fans', 'Wall Fans', 'Stand Fans', 'Water Dispensers', 'Water Pumps', 'Electric Cooker', 'Electric heaters', 'Electric Irons', 'Sewing Machine', 'Microwave Ovens', 'Geysers', 'UPS', 'Other Electronic Devices']


In [63]:
test_merge = consumption.merge(
    metadata_subset,
    on="House",
    how="left"
)

print(test_merge.columns.tolist())

['House', 'Date', 'Electricity_Consumption_kWh', 'Average_Load_kW', 'Peak_Load_kW', 'City', 'Owner/Rented', 'No. of people (Temp+Perm)', 'No. of Permanent residents', 'No. of Children (0-13)', 'No. of Adults (14-60)', 'No. of Seniors (above 60)', 'No. of temporary residents', 'Property Area (Marla)', 'Covered Area', 'No of Floors', 'Floor of Residency', 'Build year of house', 'Wapda Connection type', 'Average Ceiling Height ft', 'Ceiling Type', 'Roof Type', 'Flooring Type', 'Interior Wall', 'Exterior Wall', 'No. of rooms', 'Room Dimensions', 'Kitchen', 'Number of Washrooms', 'Number of Stores', 'Doors Type', 'Air Conditioners', 'Air Coolers', 'Refrigerators', 'Washing Machines', 'LED Bulbs', 'Tube Lights', 'Celling Fans', 'Wall Fans', 'Stand Fans', 'Water Dispensers', 'Water Pumps', 'Electric Cooker', 'Electric heaters', 'Electric Irons', 'Sewing Machine', 'Microwave Ovens', 'Geysers', 'UPS', 'Other Electronic Devices']


In [64]:
weather = pd.read_csv("Islamabad.csv")

print("Shape:", weather.shape)
print("\nColumns:")
print(weather.columns.tolist())

print("\nFirst 5 rows:")
display(weather.head())

Shape: (12456, 11)

Columns:
['datetime', 'Temperature', 'Humidity', 'Dew', 'Precipitation', 'Wind Speed', 'Wind Direction', 'Pressure', 'Solar Radiation', 'Solar Energy', 'UV Index']

First 5 rows:


,datetime,Temperature,Humidity,Dew,Precipitation,Wind Speed,Wind Direction,Pressure,Solar Radiation,Solar Energy,UV Index
0,2023-07-01,28.3,74.34,23.3,0.0,2.9,62.8,1001.0,0.0,0.0,0
1,2023-07-01 01:00:00,27.5,76.50,23.0,0.0,3.2,65.7,1000.0,0.0,0.0,0
2,2023-07-01 02:00:00,24.3,70.53,18.6,0.0,6.4,130.0,1001.3,0.0,0.0,0
3,2023-07-01 03:00:00,27.0,74.58,22.1,0.0,8.6,54.8,1001.0,0.0,0.0,0
4,2023-07-01 04:00:00,27.1,71.48,21.5,0.0,8.6,63.2,1001.0,0.0,0.0,0


In [65]:
import pandas as pd

# Load Islamabad weather data
weather = pd.read_csv("Islamabad.csv")

# Convert datetime correctly
weather["datetime"] = pd.to_datetime(
    weather["datetime"],
    format="mixed"
)

# Create Date column for merging with electricity data
weather["Date"] = weather["datetime"].dt.date

# Add city
weather["City"] = "Islamabad"

# Check the result
print("Weather shape:", weather.shape)
print("\nWeather columns:")
print(weather.columns.tolist())

display(weather.head())

Weather shape: (12456, 13)

Weather columns:
['datetime', 'Temperature', 'Humidity', 'Dew', 'Precipitation', 'Wind Speed', 'Wind Direction', 'Pressure', 'Solar Radiation', 'Solar Energy', 'UV Index', 'Date', 'City']


,datetime,Temperature,Humidity,Dew,Precipitation,Wind Speed,Wind Direction,Pressure,Solar Radiation,Solar Energy,UV Index,Date,City
0,2023-07-01 00:00:00,28.3,74.34,23.3,0.0,2.9,62.8,1001.0,0.0,0.0,0,2023-07-01,Islamabad
1,2023-07-01 01:00:00,27.5,76.50,23.0,0.0,3.2,65.7,1000.0,0.0,0.0,0,2023-07-01,Islamabad
2,2023-07-01 02:00:00,24.3,70.53,18.6,0.0,6.4,130.0,1001.3,0.0,0.0,0,2023-07-01,Islamabad
3,2023-07-01 03:00:00,27.0,74.58,22.1,0.0,8.6,54.8,1001.0,0.0,0.0,0,2023-07-01,Islamabad
4,2023-07-01 04:00:00,27.1,71.48,21.5,0.0,8.6,63.2,1001.0,0.0,0.0,0,2023-07-01,Islamabad


In [66]:
# Create daily weather features
weather_daily = (
    weather.groupby(["City", "Date"])
    .agg(
        Temperature_Avg_C=("Temperature", "mean"),
        Temperature_Min_C=("Temperature", "min"),
        Temperature_Max_C=("Temperature", "max"),
        Humidity_Avg_pct=("Humidity", "mean"),
        Dew_Avg=("Dew", "mean"),
        Precipitation_mm=("Precipitation", "sum"),
        WindSpeed_Avg=("Wind Speed", "mean"),
        Pressure_Avg=("Pressure", "mean"),
        SolarRadiation_Avg=("Solar Radiation", "mean"),
        SolarEnergy_Sum=("Solar Energy", "sum"),
        UVIndex_Avg=("UV Index", "mean")
    )
    .reset_index()
)

print("Daily weather shape:", weather_daily.shape)
display(weather_daily.head())

Daily weather shape: (519, 13)


,City,Date,Temperature_Avg_C,Temperature_Min_C,Temperature_Max_C,Humidity_Avg_pct,Dew_Avg,Precipitation_mm,WindSpeed_Avg,Pressure_Avg,SolarRadiation_Avg,SolarEnergy_Sum,UVIndex_Avg
0,Islamabad,2023-07-01,30.120833,21.4,36.3,56.889583,20.141667,0.000,7.875000,1000.441667,315.358333,27.2,3.125000
1,Islamabad,2023-07-02,31.866667,22.6,37.5,54.744583,21.241667,0.000,8.008333,1000.700000,303.495833,26.2,3.000000
2,Islamabad,2023-07-03,32.341667,22.7,39.1,54.057083,21.362500,3.285,9.579167,999.900000,292.708333,25.3,2.958333
3,Islamabad,2023-07-04,29.187500,24.1,35.1,61.527917,20.825000,17.725,11.741667,998.766667,246.762500,21.2,2.500000
4,Islamabad,2023-07-05,26.537500,23.1,29.2,71.199583,20.804167,3.192,11.595833,1000.287500,224.520833,19.3,2.166667


In [67]:
print("Weather date range:")
print(weather_daily["Date"].min(), "to", weather_daily["Date"].max())

print("\nConsumption date range:")
print(consumption["Date"].min(), "to", consumption["Date"].max())

Weather date range:
2023-07-01 to 2024-11-30

Consumption date range:
2023-11-01 00:00:00 to 2024-10-30 00:00:00


## 11. Merge electricity + household characteristics + weather

In [68]:
# Make both Date columns the same datetime type

consumption["Date"] = pd.to_datetime(consumption["Date"]).dt.normalize()
weather_daily["Date"] = pd.to_datetime(weather_daily["Date"]).dt.normalize()

print("Consumption Date type:", consumption["Date"].dtype)
print("Weather Date type:", weather_daily["Date"].dtype)

Consumption Date type: datetime64[ns]
Weather Date type: datetime64[ns]


In [69]:
# Merge electricity consumption with household metadata
model_df = consumption.merge(
    metadata_subset,
    on="House",
    how="left"
)

# Merge daily weather
model_df = model_df.merge(
    weather_daily,
    on=["City", "Date"],
    how="left"
)

# Sort the final dataset
model_df = model_df.sort_values(
    ["House", "Date"]
).reset_index(drop=True)

print("Final dataset shape:", model_df.shape)
display(model_df.head())

Final dataset shape: (716, 61)


,House,Date,Electricity_Consumption_kWh,Average_Load_kW,Peak_Load_kW,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),...,Temperature_Min_C,Temperature_Max_C,Humidity_Avg_pct,Dew_Avg,Precipitation_mm,WindSpeed_Avg,Pressure_Avg,SolarRadiation_Avg,SolarEnergy_Sum,UVIndex_Avg
0,House#41,2023-11-01,0.010500,0.63,0.63,Islamabad,Owner,6.0,6.0,2.0,...,9.8,27.4,69.213750,14.708333,0.245,7.616667,1018.425000,151.004167,13.0,1.458333
1,House#41,2023-11-02,0.008500,0.51,0.51,Islamabad,Owner,6.0,6.0,2.0,...,15.5,26.6,62.315833,11.987500,0.329,4.720833,1016.562500,165.195833,14.2,1.666667
2,House#41,2023-11-03,0.010500,0.63,0.63,Islamabad,Owner,6.0,6.0,2.0,...,8.8,26.7,61.763750,11.475000,0.000,5.650000,1013.525000,173.904167,15.0,1.750000
3,House#41,2023-11-04,0.010167,0.61,0.61,Islamabad,Owner,6.0,6.0,2.0,...,14.3,26.6,59.337083,10.541667,0.000,5.491667,1014.308333,171.891667,14.8,1.750000
4,House#41,2023-11-05,0.008333,0.50,0.50,Islamabad,Owner,6.0,6.0,2.0,...,8.4,26.3,57.937917,9.441667,0.000,4.620833,1015.170833,173.462500,15.0,1.750000


In [70]:
print("Missing temperature:",
      model_df["Temperature_Avg_C"].isna().sum())

print("Total rows:",
      len(model_df))

Missing temperature: 0
Total rows: 716


## 13. Feature engineering

We add:

- month
- day of week
- weekend
- day of year
- seasonal sine/cosine features
- yesterday's consumption
- consumption 7 days ago
- previous 7-day average consumption

The lag features are calculated separately for each house.


In [71]:
import numpy as np
import pandas as pd

# Calendar features
model_df["Year"] = pd.to_datetime(model_df["Date"]).dt.year
model_df["Month"] = pd.to_datetime(model_df["Date"]).dt.month
model_df["Day"] = pd.to_datetime(model_df["Date"]).dt.day
model_df["Day_of_Week"] = pd.to_datetime(model_df["Date"]).dt.dayofweek
model_df["Day_of_Year"] = pd.to_datetime(model_df["Date"]).dt.dayofyear

# Weekend
model_df["Is_Weekend"] = (
    model_df["Day_of_Week"] >= 5
).astype(int)

# Cyclical month features
model_df["Month_Sin"] = np.sin(
    2 * np.pi * model_df["Month"] / 12
)

model_df["Month_Cos"] = np.cos(
    2 * np.pi * model_df["Month"] / 12
)

print("Calendar features created.")
display(model_df.head())

Calendar features created.


,House,Date,Electricity_Consumption_kWh,Average_Load_kW,Peak_Load_kW,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),...,SolarEnergy_Sum,UVIndex_Avg,Year,Month,Day,Day_of_Week,Day_of_Year,Is_Weekend,Month_Sin,Month_Cos
0,House#41,2023-11-01,0.010500,0.63,0.63,Islamabad,Owner,6.0,6.0,2.0,...,13.0,1.458333,2023,11,1,2,305,0,-0.5,0.866025
1,House#41,2023-11-02,0.008500,0.51,0.51,Islamabad,Owner,6.0,6.0,2.0,...,14.2,1.666667,2023,11,2,3,306,0,-0.5,0.866025
2,House#41,2023-11-03,0.010500,0.63,0.63,Islamabad,Owner,6.0,6.0,2.0,...,15.0,1.750000,2023,11,3,4,307,0,-0.5,0.866025
3,House#41,2023-11-04,0.010167,0.61,0.61,Islamabad,Owner,6.0,6.0,2.0,...,14.8,1.750000,2023,11,4,5,308,1,-0.5,0.866025
4,House#41,2023-11-05,0.008333,0.50,0.50,Islamabad,Owner,6.0,6.0,2.0,...,15.0,1.750000,2023,11,5,6,309,1,-0.5,0.866025


In [72]:
# Previous-day consumption
model_df["lag_1_kWh"] = (
    model_df.groupby("House")["Electricity_Consumption_kWh"]
    .shift(1)
)

# Consumption from 7 days ago
model_df["lag_7_kWh"] = (
    model_df.groupby("House")["Electricity_Consumption_kWh"]
    .shift(7)
)

# Previous 7-day average consumption
model_df["rolling_7_kWh"] = (
    model_df.groupby("House")["Electricity_Consumption_kWh"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

display(
    model_df[
        [
            "House",
            "Date",
            "Electricity_Consumption_kWh",
            "lag_1_kWh",
            "lag_7_kWh",
            "rolling_7_kWh"
        ]
    ].head(10)
)

,House,Date,Electricity_Consumption_kWh,lag_1_kWh,lag_7_kWh,rolling_7_kWh
0,House#41,2023-11-01,0.010500,NaN,NaN,NaN
1,House#41,2023-11-02,0.008500,0.010500,NaN,NaN
2,House#41,2023-11-03,0.010500,0.008500,NaN,NaN
3,House#41,2023-11-04,0.010167,0.010500,NaN,NaN
4,House#41,2023-11-05,0.008333,0.010167,NaN,NaN
5,House#41,2023-11-06,0.008500,0.008333,NaN,NaN
6,House#41,2023-11-07,0.009500,0.008500,NaN,NaN
7,House#41,2023-11-08,0.008333,0.009500,0.0105,0.009429
8,House#41,2023-11-09,0.008833,0.008333,0.0085,0.009119
9,House#41,2023-11-10,0.018667,0.008833,0.0105,0.009167


In [73]:
model_df = model_df.dropna(
    subset=[
        "lag_1_kWh",
        "lag_7_kWh",
        "rolling_7_kWh"
    ]
).reset_index(drop=True)

print("Training dataset shape:", model_df.shape)

Training dataset shape: (702, 72)


In [74]:
model_df.to_csv(
    "PowerPlus_merged_training_data.csv",
    index=False
)

print("PowerPlus training dataset saved successfully.")

PowerPlus training dataset saved successfully.


## 15. Define target and predictors

Target:

**Electricity_Consumption_kWh**

We remove:

- target itself
- House ID
- Date

The original Date is replaced with engineered date variables.


In [75]:
print("Target statistics:")
print(
    model_df["Electricity_Consumption_kWh"].describe()
)

Target statistics:
count    702.000000
mean       0.012800
std        0.015572
min        0.000000
25%        0.000000
50%        0.007972
75%        0.018517
max        0.081816
Name: Electricity_Consumption_kWh, dtype: float64


In [76]:
print("Training dataset shape:", model_df.shape)
print(model_df["Electricity_Consumption_kWh"].describe())

Training dataset shape: (702, 72)
count    702.000000
mean       0.012800
std        0.015572
min        0.000000
25%        0.000000
50%        0.007972
75%        0.018517
max        0.081816
Name: Electricity_Consumption_kWh, dtype: float64


## 16. Chronological train/test split

For forecasting, we use the first **80% of dates for training** and the final **20% for testing**.

This is preferable to a random split because future observations should not be used to train the model.


In [77]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Target
target = "Electricity_Consumption_kWh"

# Sort chronologically
model_df = model_df.sort_values(
    ["Date", "House"]
).reset_index(drop=True)

# Remove target and date from features
X = model_df.drop(
    columns=[target, "Date"]
)

y = model_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (702, 70)
y shape: (702,)


## 17. Preprocessing

In [78]:
# 80% training, 20% testing
split_index = int(len(model_df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining date range:")
print(model_df["Date"].iloc[:split_index].min(),
      "to",
      model_df["Date"].iloc[:split_index].max())

print("\nTesting date range:")
print(model_df["Date"].iloc[split_index:].min(),
      "to",
      model_df["Date"].iloc[split_index:].max())

Training samples: 561
Testing samples: 141

Training date range:
2023-11-08 00:00:00 to 2024-08-21 00:00:00

Testing date range:
2024-08-21 00:00:00 to 2024-10-30 00:00:00


In [79]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

Numeric features: 57
Categorical features: 13

Categorical columns:
['House', 'City', 'Owner/Rented', 'Floor of Residency', 'Wapda Connection type', 'Ceiling Type', 'Roof Type', 'Flooring Type', 'Interior Wall', 'Exterior Wall', 'Room Dimensions', 'Doors Type', 'Other Electronic Devices']


## 18. Decision Tree Regressor

In [80]:
# Numeric preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Decision Tree
dt_model = DecisionTreeRegressor(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

# Complete pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", dt_model)
    ]
)

print("Decision Tree pipeline created successfully.")

Decision Tree pipeline created successfully.


In [81]:
# Train Decision Tree
model.fit(X_train, y_train)

print("Decision Tree model trained successfully.")

Decision Tree model trained successfully.


## 19. Predict and evaluate

In [82]:
# Predictions
y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

First 10 predictions:
[0.01795238 0.         0.0324383  0.         0.02466622 0.
 0.01527698 0.         0.04405253 0.        ]


In [83]:
# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("Model Performance")
print("-----------------")
print(f"MAE  : {mae:.6f} kWh")
print(f"RMSE : {rmse:.6f} kWh")
print(f"R²   : {r2:.4f}")

Model Performance
-----------------
MAE  : 0.000154 kWh
RMSE : 0.000383 kWh
R²   : 0.9992


## 20. Actual vs predicted